In [1]:
# CELL 1 — Configuration
# Run these cells in order in /home/shatakd/Inward_pipeline.
# Uses your existing images2splat environment and installed H100 trainer.
# This is the PRACTICE pipeline: RGB images → reconstruction → Gaussian PLY.
# It does not use LiDAR or produce verified metric/navigation alignment.

from pathlib import Path
import getpass
import json
import shlex
import subprocess

PROJECT = Path("/home/shatakd/Inward_pipeline")
CAPTURE = PROJECT / "capture_hallway"
WORK = PROJECT / "runs/hallway_images_001"

ENVIRONMENT = Path("/home/shatakd/.conda/envs/images2splat")
PYTHON = ENVIRONMENT / "bin/python"
REPOSITORY = Path("/home/shatakd/gaussian-splatting")

DATASET = WORK / "practice_dataset"
OUTPUT = WORK / "practice_training_30k_fullres"

ITERATIONS = 30000
RESOLUTION = 1
DENSIFY_UNTIL = 15000

# Explicitly accepted for this practice capture: 121/151 images = 80.1%.
# For a new capture, choose a fresh WORK and OUTPUT directory.
MIN_REGISTRATION_FRACTION = 0.80

FINAL_PLY = (
    OUTPUT / "point_cloud"
    / f"iteration_{ITERATIONS}"
    / "point_cloud.ply"
)

CONFIG = WORK / "notebook_config.json"
STATE_FILE = WORK / "notebook_jobs.json"

for required in [
    PROJECT / "pipeline.py",
    PROJECT / "cluster_env.sh",
    PYTHON,
    REPOSITORY / "train.py",
]:
    if not required.is_file():
        raise FileNotFoundError(required)

images = sorted(
    p for p in (CAPTURE / "rgb").iterdir()
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
if len(images) < 3:
    raise RuntimeError("Put your JPG/PNG images inside capture_hallway/rgb.")

WORK.mkdir(parents=True, exist_ok=True)

config = {
    "capture": str(CAPTURE),
    "work": str(WORK),
    "gs_repo": str(REPOSITORY),
    "colmap": str(ENVIRONMENT / "bin/colmap"),
    "camera_model": "OPENCV",
    "single_camera": False,
    "matcher": "exhaustive",
    "min_registration_fraction": MIN_REGISTRATION_FRACTION,
    "iterations": ITERATIONS,
    "resolution": RESOLUTION,
}

if CONFIG.exists() and json.loads(CONFIG.read_text()) != config:
    raise RuntimeError(
        "This run already has a different configuration. "
        "Use a new WORK directory instead of overwriting it."
    )

CONFIG.write_text(json.dumps(config, indent=2) + "\n")

print("Images:", len(images))
print("Dataset:", DATASET)
print("Training output:", OUTPUT)
print("Final PLY:", FINAL_PLY)

Images: 151
Dataset: /home/shatakd/Inward_pipeline/runs/hallway_images_001/practice_dataset
Training output: /home/shatakd/Inward_pipeline/runs/hallway_images_001/practice_training_30k_fullres
Final PLY: /home/shatakd/Inward_pipeline/runs/hallway_images_001/practice_training_30k_fullres/point_cloud/iteration_30000/point_cloud.ply


In [2]:
# CELL 2 — Job helpers and environment checks
# Job IDs are saved to disk, so monitoring survives notebook restarts.

def run_command(command):
    result = subprocess.run(
        [str(value) for value in command],
        cwd=PROJECT,
        capture_output=True,
        text=True,
    )
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout.strip()


def read_state():
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}


def save_state(state):
    temporary = STATE_FILE.with_suffix(".tmp")
    temporary.write_text(json.dumps(state, indent=2) + "\n")
    temporary.replace(STATE_FILE)


def queue_jobs():
    output = run_command([
        "squeue", "-h", "-u", getpass.getuser(), "-o", "%i|%j"
    ])
    return dict(
        line.strip().split("|", 1)
        for line in output.splitlines()
        if "|" in line
    )


def write_batch(name, body, gpu=False):
    header = [
        "#!/bin/bash",
        f"#SBATCH --job-name={name}",
        "#SBATCH --nodes=1",
        "#SBATCH --ntasks=1",
        f"#SBATCH --cpus-per-task={4 if gpu else 8}",
        "#SBATCH --mem=32G",
        "#SBATCH --time=04:00:00",
        "#SBATCH --export=ALL",
        f"#SBATCH --output={PROJECT}/{name}-%j.log",
    ]

    if gpu:
        header.append("#SBATCH --gpus=h100:1")

    environment = f"""
set -eo pipefail
set +u
cd {shlex.quote(str(PROJECT))}
source {shlex.quote(str(PROJECT / "cluster_env.sh"))}
"""
    if gpu:
        environment += "module load cuda/11.8.0\n"

    environment += f"""
set -u
export SLURM_EXPORT_ENV=ALL
export PATH={shlex.quote(str(ENVIRONMENT / "bin"))}:"$PATH"
export QT_QPA_PLATFORM=offscreen
export PYTHONUNBUFFERED=1
export OMP_NUM_THREADS="${{SLURM_CPUS_PER_TASK:-4}}"
"""

    script = WORK / f"{name}.sbatch"
    script.write_text("\n".join(header) + "\n" + environment + body + "\n")
    run_command(["bash", "-n", script])
    return script


def submit_once(stage, name, script, dependency=None):
    state = read_state()
    active = queue_jobs()

    # Reuse a submission recorded by this notebook.
    previous = state.get(stage, {}).get("job_id")
    if previous in active:
        print(f"{stage}: already queued/running as {previous}")
        return previous

    # Also detect jobs submitted by the previous notebook.
    matches = [job for job, job_name in active.items() if job_name == name]
    if len(matches) > 1:
        raise RuntimeError(f"Multiple {name} jobs exist: {matches}")
    if matches:
        job_id = matches[0]
        print(f"{stage}: using existing job {job_id}")
    else:
        if previous:
            raise RuntimeError(
                f"Previous {stage} job {previous} is no longer active. "
                "Inspect its log in Cell 5 before retrying."
            )

        command = ["sbatch", "--parsable", "--export=ALL"]
        if dependency:
            command.append(f"--dependency=afterok:{dependency}")
        command.append(str(script))

        response = run_command(command)
        job_id = response.split(";")[0].strip()
        if not job_id.isdigit():
            raise RuntimeError(f"Unexpected submission response: {response}")
        print(f"{stage}: submitted job {job_id}")

    state[stage] = {
        "job_id": job_id,
        "log": str(PROJECT / f"{name}-{job_id}.log"),
    }
    save_state(state)
    return job_id


def dataset_ready():
    required = [
        DATASET / "images",
        DATASET / "sparse/0/cameras.bin",
        DATASET / "sparse/0/images.bin",
        DATASET / "sparse/0/points3D.bin",
        DATASET / "practice_metadata.json",
    ]
    return all(path.exists() for path in required)


print(run_command([
    PYTHON, "-c",
    "import numpy, scipy, PIL, plyfile; "
    "print('Pipeline Python dependencies are available.')"
]))

Pipeline Python dependencies are available.


In [3]:
# CELL 3 — Prepare the practice dataset
# Reuses your existing reconstruction and undistorted images when available.
# New captures run through pipeline.py; failed partial stages are not overwritten.

worker = WORK / "prepare_notebook_dataset.py"

worker.write_text(
    "import sys\n"
    f"sys.path.insert(0, {str(PROJECT)!r})\n"
    "from pathlib import Path\n"
    "import json\n"
    "import pipeline\n"
    f"CONFIG = Path({str(CONFIG)!r})\n"
    f"DATASET = Path({str(DATASET)!r})\n"
    + r'''
p = pipeline.Pipeline(CONFIG)
manifest_path = p.w / "capture_manifest.json"

if not manifest_path.exists():
    p.prepare()

manifest = pipeline.read_json(manifest_path)

# Reject reuse if the source image set or image bytes have changed.
source = Path(p.c["capture"]) / "rgb"
current = {
    f.name: f for f in source.iterdir()
    if f.is_file() and f.suffix.lower() in {".jpg", ".jpeg", ".png"}
}
expected = {frame["image"]: frame for frame in manifest["frames"]}

if set(current) != set(expected):
    raise RuntimeError("Input image set changed. Use a fresh WORK directory.")

for name, frame in expected.items():
    if pipeline.sha(current[name]) != frame["sha256"]:
        raise RuntimeError(f"Image changed: {name}. Use a fresh WORK directory.")

report_path = p.w / "registration.json"
if not report_path.exists():
    p.sfm()

report = pipeline.read_json(report_path)
fraction = report["registered"] / len(manifest["frames"])
print(f"Registered: {report['registered']}/{len(manifest['frames'])} "
      f"({fraction:.1%})", flush=True)

if fraction < p.c["min_registration_fraction"]:
    raise RuntimeError("Registration is below the practice threshold.")

required = [
    DATASET / "images",
    DATASET / "sparse/0/cameras.bin",
    DATASET / "sparse/0/images.bin",
    DATASET / "sparse/0/points3D.bin",
    DATASET / "practice_metadata.json",
]

if all(path.exists() for path in required):
    print("Existing practice dataset verified.", flush=True)
else:
    if DATASET.exists():
        raise RuntimeError(
            "Incomplete practice_dataset exists. Inspect/archive it before retrying."
        )

    model = p.w / "sfm/sparse" / str(report["selected"])
    p.colmap(
        "image_undistorter",
        [
            "--image_path", p.w / "input_images",
            "--input_path", model,
            "--output_path", DATASET,
            "--output_type", "COLMAP",
        ],
        "practice_undistort",
    )

    sparse = DATASET / "sparse"
    for name in ["cameras.bin", "images.bin", "points3D.bin"]:
        if not (sparse / name).is_file():
            raise FileNotFoundError(sparse / name)

    target = sparse / "0"
    target.mkdir()
    for item in list(sparse.iterdir()):
        if item.is_file():
            item.rename(target / item.name)

    pipeline.write_json(DATASET / "practice_metadata.json", {
        "purpose": "Practice visual reconstruction only",
        "units": "arbitrary reconstruction units",
        "metric_alignment_applied": False,
        "navigation_verified": False,
        "registered_images": report["registered"],
        "input_images": len(manifest["frames"]),
    })

print("PRACTICE DATASET READY:", DATASET, flush=True)
'''
)

body = (
    f"srun --export=ALL {shlex.quote(str(PYTHON))} "
    f"{shlex.quote(str(worker))}\n"
)
prepare_script = write_batch("inward-prepare", body)

# Existing completed datasets need no new preparation job.
# Still verify that the source images match the retained capture manifest.
if dataset_ready():
    manifest = json.loads((WORK / "capture_manifest.json").read_text())
    expected_names = {frame["image"] for frame in manifest["frames"]}
    if expected_names != {image.name for image in images}:
        raise RuntimeError("Image set changed. Use a fresh WORK directory.")

    # This performs file-integrity checks only; it reuses the ready dataset.
    print(run_command([PYTHON, worker]))
    prepare_job_id = None
else:
    prepare_job_id = submit_once(
        "prepare", "inward-prepare", prepare_script
    )

Registered: 121/151 (80.1%)
Existing practice dataset verified.
PRACTICE DATASET READY: /home/shatakd/Inward_pipeline/runs/hallway_images_001/practice_dataset


In [4]:
# CELL 5 — Monitor progress
# Rerun this cell whenever you want an update.
# No submissions, cancellations, or installations occur here.

state = read_state()
active = queue_jobs()

for stage, info in state.items():
    job_id = info["job_id"]
    print(f"\n=== {stage.upper()} — job {job_id} ===")

    if job_id in active:
        result = subprocess.run(
            ["squeue", "-j", job_id],
            capture_output=True, text=True
        )
        print(result.stdout)
    else:
        result = subprocess.run(
            [
                "sacct", "-j", job_id,
                "--format=JobID,State,ExitCode",
                "--noheader",
            ],
            capture_output=True, text=True
        )
        print(result.stdout or result.stderr or "Accounting not yet available.")

    log = Path(info["log"])
    if log.exists():
        # Read only the last 32 KB of potentially large progress logs.
        with log.open("rb") as stream:
            stream.seek(max(0, log.stat().st_size - 32768))
            text = stream.read().decode("utf-8", errors="replace")

        lines = text.replace("\r", "\n").splitlines()
        print("\n".join(lines[-25:]))
    else:
        print("Log not created yet.")

if FINAL_PLY.exists():
    print("\nFinal PLY exists:", FINAL_PLY)
    print(f"Size: {FINAL_PLY.stat().st_size / 1024**2:.1f} MiB")
else:
    print("\nFinal PLY is not available yet.")

if not state:
    print("No jobs submitted by this cleaned notebook.")


Final PLY exists: /home/shatakd/Inward_pipeline/runs/hallway_images_001/practice_training_30k_fullres/point_cloud/iteration_30000/point_cloud.ply
Size: 181.5 MiB
No jobs submitted by this cleaned notebook.


In [5]:
# CELL 6 — Validate the final Gaussian PLY and show its download location
# Run after training finishes. This does not modify the PLY.

if any(name == "splat-quality" for name in queue_jobs().values()):
    print("Training is still active. Wait before validating/downloading.")
elif not FINAL_PLY.is_file():
    print("Final PLY is not ready. Check Cell 5.")
else:
    validation_code = (
        "import sys, json; "
        f"sys.path.insert(0, {str(PROJECT)!r}); "
        "from pipeline import validate_ply; "
        f"print(json.dumps(validate_ply({str(FINAL_PLY)!r}), indent=2))"
    )
    print(run_command([PYTHON, "-c", validation_code]))

    print("\nDownload this file through Jupyter's file browser:")
    print(FINAL_PLY)
    print("\nRight-click point_cloud.ply → Download.")
    print("Open it in https://superspl.at/editor")
    print("\nPractice output only: no LiDAR, metric scale, or verified navigation.")

{
  "gaussians": 767415,
  "fields": 62,
  "bytes": 190320451,
  "sha256": "97a005a8fd94a57ef74e534610cae7da0809f5448bb8c85325865f8b7dccd644"
}

Download this file through Jupyter's file browser:
/home/shatakd/Inward_pipeline/runs/hallway_images_001/practice_training_30k_fullres/point_cloud/iteration_30000/point_cloud.ply

Right-click point_cloud.ply → Download.
Open it in https://superspl.at/editor

Practice output only: no LiDAR, metric scale, or verified navigation.
